<a href="https://colab.research.google.com/github/Erjg1012/sales_targeting-quota_setting/blob/main/Enfoque_Geoespacial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Cuotas con un enfoque geoespacial.
Para abordar el Enfoque Geográfico y de Expansión (Market Potential) utilizando el Olist Brazilian E-Commerce Dataset, el objetivo cambia radicalmente en comparación con el método tradicional. Aquí no solo nos importa cuánto se vendió antes en un lugar, sino cuánto potencial tiene un territorio que no está siendo aprovechado.
En este enfoque, la cuota no se calcula sumando un porcentaje fijo a la venta anterior, sino identificando brechas de mercado (market gaps). Si una región tiene muchos clientes potenciales o alta densidad de población pero bajas ventas, la cuota de crecimiento debe ser más agresiva.

In [2]:
import pandas as pd

# 1. Cargar los datasets indispensables
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

# 2. Unir ítems con órdenes usando 'order_id' para tener el valor de la venta
df_ventas = pd.merge(items, orders, on='order_id', how='inner')

# 3. Unir el resultado con los clientes usando 'customer_id' para obtener la ubicación ('customer_state')
df_final_geo = pd.merge(df_ventas, customers, on='customer_id', how='inner')

# Ahora ya tienes en 'df_final_geo' el estado del cliente y el precio pagado para calcular el potencial.
print(df_final_geo[['order_id', 'price', 'customer_state']].head())

                           order_id   price customer_state
0  00010242fe8c5a6d1ba2dd792cb16214   58.90             RJ
1  00018f77f2f0320c557190d7a144bdd3  239.90             SP
2  000229ec398224ef6ca0657da4fc703e  199.00             MG
3  00024acbcdf0a6daa1e931b038114c75   12.99             SP
4  00042b26cf59d7ce69dfabb4e55b4fd9  199.90             SP


Nota: Deja fuera de este módulo las tablas de reviews, payments y products, ya que solo añadirían peso a la memoria de tu Colab y no aportan al cálculo del potencial territorial.

##Cálculo del Índice de Potencial de Mercado (MPI)
Para fijar cuotas geográficas sin usar fuentes externas todavía, podemos usar una métrica interna: la densidad de órdenes vs. el ticket promedio. Sin embargo, la forma profesional de hacerlo es cruzarlo con el tamaño del mercado.

Como aproximación dentro del dataset, calcularemos la Penetración de Mercado Relativa. Asumiremos que los estados con mejor desempeño actual definen el "techo" de lo que se podría vender en proporción a su número de clientes únicos.

In [4]:
import numpy as np

# 1. Agrupar datos por estado para calcular métricas clave
df_geo = df_final_geo.groupby('customer_state').agg(
    Ventas_Totales=('price', 'sum'),
    Clientes_Unicos=('customer_unique_id', 'nunique'),
    Total_Ordenes=('order_id', 'count')
).reset_index()

# 2. Calcular el Ticket Promedio por estado
df_geo['Ticket_Promedio'] = df_geo['Ventas_Totales'] / df_geo['Total_Ordenes']

# 3. Crear un indicador de "Oportunidad" (Market Gap)
# Si un estado tiene muchos clientes únicos pero un ticket promedio bajo en comparación con la media nacional,
# significa que hay potencial de expansión en volumen o valor de los productos.
ticket_medio_nacional = df_geo['Ticket_Promedio'].median()

# Definimos el factor de oportunidad geográfica
df_geo['Factor_Oportunidad'] = np.where(
    df_geo['Ticket_Promedio'] < ticket_medio_nacional,
    (ticket_medio_nacional / df_geo['Ticket_Promedio']),
    1.0
)

##Asignación de Cuotas Dinámicas por Territorio
En lugar de aplicar un 15% de aumento plano a todos los estados (como en el método tradicional), usaremos el Factor_Oportunidad para redistribuir un objetivo global de expansión de la empresa.

Si la empresa quiere vender 1,000,000 de reales más el próximo trimestre, los estados con mayor Factor de Oportunidad (mayor brecha de mercado) absorberán una proporción más alta de la cuota.

In [5]:
# Supongamos que el objetivo de crecimiento total de la empresa para el próximo periodo es de 500,000 RS
objetivo_crecimiento_global = 500000

# Calculamos un peso ponderado que combine el volumen actual del estado y su potencial de oportunidad
df_geo['Peso_Asignacion'] = df_geo['Ventas_Totales'] * df_geo['Factor_Oportunidad']
peso_total = df_geo['Peso_Asignacion'].sum()

# Distribuimos la cuota de crecimiento global proporcionalmente
df_geo['Cuota_Incremento_Geografica'] = (df_geo['Peso_Asignacion'] / peso_total) * objetivo_crecimiento_global

# La nueva cuota objetivo final por estado será su venta histórica más el incremento por potencial
df_geo['Nueva_Cuota_Total'] = df_geo['Ventas_Totales'] + df_geo['Cuota_Incremento_Geografica']

# Ver el resultado ordenado por los estados donde se asignó más cuota debido a su potencial
df_resultado_geo = df_geo[['customer_state', 'Ventas_Totales', 'Factor_Oportunidad', 'Cuota_Incremento_Geografica', 'Nueva_Cuota_Total']]
print(df_resultado_geo.sort_values(by='Factor_Oportunidad', ascending=False).head())

   customer_state  Ventas_Totales  Factor_Oportunidad  \
25             SP      5202955.05            1.326981   
17             PR       683083.76            1.222716   
22             RS       750304.02            1.209169   
10             MG      1585308.03            1.205052   
7              ES       275037.31            1.193535   

    Cuota_Incremento_Geografica  Nueva_Cuota_Total  
25                209184.491386       5.412140e+06  
17                 25305.464405       7.083892e+05  
22                 27487.730064       7.777918e+05  
10                 57880.739055       1.643189e+06  
7                   9945.841062       2.849832e+05  
